# Milestone 2: RAG Pipeline Explorations
This notebook documents our initial experiments with setting up a local LLM (Llama 3.2 via Ollama), testing our custom Semantic and BM25 retrievers, and building our first LangChain (LCEL) pipelines before moving them to production Python scripts.

In [ ]:
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path(os.getcwd()).parent))

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Initializing Llama 3.2...")
llm = ChatOllama(model="llama3.2", temperature=0)

response = llm.invoke("What is a moisturizer? Explain in one sentence.")
print("\nLLM Response:")
print(response.content)

Initializing Llama 3.2...

LLM Response:
A moisturizer is a topical product that helps to hydrate and soften the skin, reducing dryness, irritation, and fine lines, while also providing various benefits such as protecting against environmental stressors and promoting skin elasticity.


### Testing the Retriever
Before hooking the search engine up to the LLM, we need to verify that our custom FAISS index (built in Milestone 1) can be successfully loaded and queried within this environment.

In [ ]:
from src.utils import load_corpus_parquet
from src.semantic import semantic_search, load_semantic_artifacts
from sentence_transformers import SentenceTransformer

root = Path(os.getcwd()).parent
_, corpus = load_corpus_parquet(root / "data/processed/products.parquet")
faiss_index, doc_ids, config = load_semantic_artifacts(root / "artifacts")
st_model = SentenceTransformer(config["model_name"])

test_query = "sunscreen for sensitive skin"
results = semantic_search(test_query, st_model, faiss_index, doc_ids, corpus, top_k=2)

print(f"Top 2 Results for '{test_query}':\n")
for i, r in enumerate(results, 1):
    print(f"Result {i}: {r.get('title', 'N/A')}")
    print(f"Price: ${r.get('price', 'N/A')}")
    print(f"Review Snippet: {r.get('review_text', '')[:100]}...\n")

Top 2 Results for 'sunscreen for sensitive skin':

Result 1: MDSolarSciences Mineral KidCreme SPF 50 – Gentle Water-Resistant Sunscreen for Kids - Hypoallergenic Broad Spectrum Face and Body UV Protection – Fragrance Free Vegan Zinc Oxide, 3.4 Oz
Price: $35.0
Review Snippet: Some people think they have sensitive skin - they just don't know what sensitive skin really is. I h...

Result 2: Rohto SKIN AQUA UV Super Moisture Gel 110g SPF50 / PA
Price: $nan
Review Snippet: This gel sunscreen is perfect for sensitive skin like me. I love it more than ANESSA one.<br />I hig...



### Exploring the RAG Chain (LCEL)
Now we combine the retriever and the LLM using LangChain Expression Language.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def build_context(docs: list[dict]) -> str:
    context_str = ""
    for i, doc in enumerate(docs, 1):
        asin = doc.get("parent_asin") or doc.get("asin", "N/A")
        title = doc.get("title", "No Title")
        price = doc.get("price")
        price_str = f"${price}" if price is not None and str(price).lower() != "nan" else "Price not listed"
        review = doc.get("review_text", "No review")
        
        context_str += f"ASIN: {asin}\nTitle: {title}\nPrice: {price_str}\nReview: {review}\n\n"
    return context_str

def retrieve_docs(query: str) -> list[dict]:
    return semantic_search(query, st_model, faiss_index, doc_ids, corpus, top_k=3)

SYSTEM_PROMPT = """
You are a factual Amazon assistant. Answer using ONLY the context below.
Always cite the ASIN. Do not invent prices.
Context:
{context}

Question: {question}
Answer:
"""
prompt_template = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

experimental_chain = (
    {
        "context": RunnableLambda(retrieve_docs) | build_context,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("Running RAG Pipeline Test...")
print(experimental_chain.invoke("What is a good sunscreen for sensitive skin?"))

Running RAG Pipeline Test...
Based on the provided context, I would recommend the Rohto SKIN AQUA UV Super Moisture Gel 110g SPF50 / PA (ASIN: B084C5TMT3) as a good sunscreen option for sensitive skin. It has received positive reviews from users who have found it to be suitable for their sensitive skin types, with one reviewer stating that they "love" it more than another brand.


### Conclusion
The local model successfully runs, and the LangChain pipeline properly formats and passes retrieved documents to the LLM. 

**Note on Hybrid Retrieval:** Following these initial semantic explorations, we integrated BM25 to build a Hybrid RRF Retriever. The final, optimized pipeline (including advanced prompt variants) was migrated to `src/hybrid_rag_pipeline.py` and deployed via `app/app.py`.